# Prompt Engineering Techniques: Constrained Decoding

Source slide: `slides/prompt-engineering/04.5_Prompt_Engineering_Techniques.Constrained_Decoding.md`

Each example below demonstrates one constrained-decoding technique from the slide.



## Prerequisites

- From the repository root, install all notebook and test prerequisites: `pip install -r requirements-dev.txt`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a failure cell and an improved cell that you can run independently with the real GitHub Copilot SDK.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Grounding

**Failure mode:** Grounding keeps the answer tied to verifiable evidence instead of model memory.

**Failure test:** The failure prompt asks a policy question with no source material, so the model can only rely on prior knowledge or guesses.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Tell the model to answer only from the supplied material and to admit when the answer is missing.

**Improved example:** The improved prompt includes the source excerpt and the fallback behavior, which reduces hallucination and makes the answer auditable. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'What is our policy on refunding annual contracts?'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks a policy question with no source material, so the model can only rely on prior knowledge or guesses.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Based only on the provided policy excerpt, answer the question. If the answer is not in the excerpt, say “I cannot find this information.”\n\nPolicy excerpt:\n- Annual contracts may be refunded within 30 days of purchase.\n- Monthly plans may be refunded within 7 days of purchase.\n\nQuestion: What is our policy on refunding annual contracts?'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt includes the source excerpt and the fallback behavior, which reduces hallucination and makes the answer auditable.'
show_block("🔧 Why this is better", fix_explanation)


## Logit Biasing

**Failure mode:** Logit biasing changes token probabilities directly, which is useful when prompt wording alone does not fully control the vocabulary.

**Failure test:** The failure run asks for preferred terminology but has no hard mechanism for suppressing unwanted tokens, so the model may still choose the wrong term.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Pair prompt guidance with token-level bias controls when the API exposes them.

**Improved example:** The improved prompt makes the desired vocabulary explicit, and the configuration note tells you where logit bias would be added in a provider that supports it. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.

**Configuration note:** In providers that support logit bias, increase the probability of the preferred token and suppress banned tokens, then rerun the improved prompt.


In [ ]:
config_note = 'In providers that support logit bias, increase the probability of the preferred token and suppress banned tokens, then rerun the improved prompt.'
show_block("⚙️ Configuration note", config_note)
failure_prompt = 'Describe this assistant as helpful, supportive, and never rude.'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure run asks for preferred terminology but has no hard mechanism for suppressing unwanted tokens, so the model may still choose the wrong term.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
config_note = 'In providers that support logit bias, increase the probability of the preferred token and suppress banned tokens, then rerun the improved prompt.'
show_block("⚙️ Configuration note", config_note)
improved_prompt = 'Describe this assistant in one sentence using the word “helpful” and avoiding any toxic or insulting wording.'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt makes the desired vocabulary explicit, and the configuration note tells you where logit bias would be added in a provider that supports it.'
show_block("🔧 Why this is better", fix_explanation)


## Structured Output

**Failure mode:** Structured output makes responses predictable enough for downstream programs to parse.

**Failure test:** The failure prompt asks for extraction but does not define a schema, so the answer can mix prose and data.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Provide an explicit machine-readable schema and require the model to answer only in that structure.

**Improved example:** The improved prompt defines the JSON schema and forbids extra prose, which makes the output easier to validate automatically. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Extract the contact details from this text: “Jamie Rivera can be reached at jamie@example.com and 555-0101.”'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for extraction but does not define a schema, so the answer can mix prose and data.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Extract the contact details from this text and return JSON only in this schema: {"name": string, "email": string, "phone": string}. Text: “Jamie Rivera can be reached at jamie@example.com and 555-0101.”'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt defines the JSON schema and forbids extra prose, which makes the output easier to validate automatically.'
show_block("🔧 Why this is better", fix_explanation)
